# NewtonAI Subtitle Generator and Summarizer

## Project Objective

This notebook documents the final backend workflow for the NewtonAI Subtitle Generator and Summarizer project. The goal is to process educational lecture videos and generate:

- extracted audio files,
- Whisper transcripts,
- `.srt` subtitle files,
- concise FLAN-T5 summaries,
- WER and ROUGE evaluation results.

The notebook is aligned with the current final backend implementation. It is intentionally lightweight for review: it explains the pipeline, shows safe imports and paths, displays outputs/results, and provides optional commands for running the full backend pipeline manually.

## Backend Pipeline Overview

The backend pipeline processes each lecture video through these stages:

1. **Audio extraction** with FFmpeg.
2. **Speech transcription** with Whisper `base`.
3. **Transcript-to-SRT generation** from Whisper timestamped segments.
4. **Transcript preprocessing** with `summary_preprocessor.py` to clean transcript text and extract lecture points.
5. **FLAN-T5 summarization** using compact preprocessed lecture notes.
6. **Evaluation** using WER and ROUGE.

The main command-line entry point is:

```bash
cd C:\subtitle_generator_summarizer\backend
python run_pipeline.py lecture_1.mp4
```

The same workflow can be run for `lecture_2.mp4` and `lecture_3.mp4`, or for all videos by running `python run_pipeline.py` without a filename.

## Tools and Models Used

| Purpose | Tool or Model |
| --- | --- |
| Audio extraction | FFmpeg |
| Speech-to-text transcription | OpenAI Whisper `base` |
| Subtitle generation | Custom Python SRT conversion code |
| Summarization | `google/flan-t5-base` |
| Transcript preprocessing | `summary_preprocessor.py` |
| WER evaluation | `jiwer` |
| ROUGE evaluation | `rouge-score` |
| Reporting and CSV handling | `pandas` |

## Backend Module Responsibilities

| Module | Responsibility |
| --- | --- |
| `audio_extractor.py` | Extracts `.wav` audio from input lecture videos using FFmpeg. |
| `transcriber.py` | Loads Whisper `base`, transcribes audio, and returns full text plus timestamped segments. |
| `srt_generator.py` | Converts Whisper segments into `.srt` subtitle files with start/end timestamps. |
| `summary_preprocessor.py` | Cleans transcript text, removes filler/speaker-style wording, extracts important lecture points, and builds compact FLAN-T5 input. |
| `summarizer.py` | Loads FLAN-T5, generates summaries, validates output, applies deterministic fallback if needed, saves summary files. |
| `evaluator.py` | Computes WER and ROUGE metrics when reference files are available. |
| `pipeline.py` | Orchestrates audio extraction, transcription, subtitle generation, summarization, and evaluation for one video. |
| `run_pipeline.py` | Command-line runner for one video or all videos in `backend/data/raw_videos/`. |

## Project Path Setup

The following lightweight cell sets up project paths and imports backend configuration. It does not run the heavy pipeline.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BACKEND_DIR = PROJECT_ROOT / "backend"

if str(BACKEND_DIR) not in sys.path:
    sys.path.append(str(BACKEND_DIR))

from app.config import (
    RAW_VIDEOS_DIR,
    AUDIO_OUTPUT_DIR,
    TRANSCRIPT_OUTPUT_DIR,
    SUBTITLE_OUTPUT_DIR,
    SUMMARY_OUTPUT_DIR,
    EVALUATION_OUTPUT_DIR,
    REFERENCE_TRANSCRIPTS_DIR,
    REFERENCE_SUMMARIES_DIR,
)

print("Project root:", PROJECT_ROOT)
print("Backend directory:", BACKEND_DIR)
print("Raw videos directory:", RAW_VIDEOS_DIR)

## Input Videos

The current backend run used three lecture videos:

- `lecture_1.mp4`
- `lecture_2.mp4`
- `lecture_3.mp4`

They are expected under:

```text
backend/data/raw_videos/
```

In [ ]:
video_files = sorted(RAW_VIDEOS_DIR.glob("*.mp4"))

print(f"Videos found: {len(video_files)}")
for video_path in video_files:
    print("-", video_path.name)

## Audio Extraction

Audio extraction is handled by `backend/app/audio_extractor.py`. It uses FFmpeg to convert each input video into a `.wav` file. The generated audio files are stored in:

```text
backend/outputs/audio/
```

Expected final outputs:

- `lecture_1.wav`
- `lecture_2.wav`
- `lecture_3.wav`

## Whisper Transcription

Transcription is handled by `backend/app/transcriber.py`. The backend uses Whisper `base`, configured in `backend/app/config.py`.

For each `.wav` file, Whisper generates:

- full transcript text,
- timestamped transcript segments.

Transcript files are saved in:

```text
backend/outputs/transcripts/
```

## Transcript-to-SRT Generation

Subtitle generation is handled by `backend/app/srt_generator.py`. Whisper timestamped segments are converted into standard `.srt` subtitle blocks containing:

- subtitle index,
- start timestamp,
- end timestamp,
- subtitle text.

Subtitle files are saved in:

```text
backend/outputs/subtitles/
```

## FLAN-T5 Summarization

Summarization is handled by `backend/app/summarizer.py`, using `google/flan-t5-base`.

The summarizer:

- receives transcript text from the pipeline,
- uses preprocessed lecture points instead of blindly sending the full raw transcript,
- validates model output for length, prompt leakage, first-person wording, and copied transcript fragments,
- saves the final summary file,
- returns the output path to the pipeline.

Summary files are saved in:

```text
backend/outputs/summaries/
```

## Summary Preprocessing

The current final backend includes `backend/app/summary_preprocessor.py` to improve summary quality before FLAN-T5 generation.

It performs the following tasks:

- cleans transcript text,
- removes filler and speaker-style wording,
- removes source/channel language when possible,
- splits transcript text into candidate sentences,
- scores and selects important lecture points,
- extracts key concepts,
- builds compact model input for FLAN-T5,
- provides a deterministic fallback when model output is weak.

This helps the final summaries stay academic, concise, lecture-specific, and less like raw transcript fragments.

## Optional Pipeline Run

The full pipeline is computationally heavy because it runs Whisper and FLAN-T5. For final review, run it manually from Anaconda Prompt if regeneration is needed:

```bash
conda activate subtitleenv
cd C:\subtitle_generator_summarizer\backend
python run_pipeline.py lecture_1.mp4
python run_pipeline.py lecture_2.mp4
python run_pipeline.py lecture_3.mp4
```

The notebook does not automatically trigger this heavy run.

In [ ]:
# Optional example only. Uncomment one line if you intentionally want to run the backend pipeline.
# from app.pipeline import process_single_video
# result = process_single_video(RAW_VIDEOS_DIR / "lecture_1.mp4")
# result

## Final Output Files

The final backend run produced:

- 3 audio `.wav` files,
- 3 transcript `.txt` files,
- 3 subtitle `.srt` files,
- 3 summary `.txt` files,
- `evaluation_report.csv`.

The next lightweight cell lists the current generated outputs without modifying them.

In [ ]:
output_groups = {
    "Audio": AUDIO_OUTPUT_DIR.glob("*.wav"),
    "Transcripts": TRANSCRIPT_OUTPUT_DIR.glob("*_transcript.txt"),
    "Subtitles": SUBTITLE_OUTPUT_DIR.glob("*.srt"),
    "Summaries": SUMMARY_OUTPUT_DIR.glob("*_summary.txt"),
    "Evaluation": EVALUATION_OUTPUT_DIR.glob("evaluation_report.csv"),
}

for label, files in output_groups.items():
    print(f"\n{label}")
    for file_path in sorted(files):
        print("-", file_path.name)

## Evaluation Methodology

The backend computes:

- **WER** using `jiwer` to compare generated transcripts with reference transcripts.
- **ROUGE-1 F1** using unigram overlap between generated and reference summaries.
- **ROUGE-2 F1** using bigram overlap.
- **ROUGE-L F1** using longest common subsequence overlap.

Reference files are stored in:

```text
backend/data/reference_transcripts/
backend/data/reference_summaries/
```

## Honest Evaluation Note

WER was computed using pseudo-reference transcripts derived from generated transcripts due to absence of official human transcripts. ROUGE was computed against manually written reference summaries.

Because pseudo-reference transcripts are copied from generated transcripts, WER values of `0.0` should not be interpreted as official human-reference transcription accuracy.

## Current Evaluation Results

| Video Name | WER | ROUGE-1 F1 | ROUGE-2 F1 | ROUGE-L F1 |
| --- | ---: | ---: | ---: | ---: |
| `lecture_1.mp4` | 0.0 | 0.5437 | 0.2376 | 0.3689 |
| `lecture_2.mp4` | 0.0 | 0.4957 | 0.1043 | 0.3077 |
| `lecture_3.mp4` | 0.0 | 0.8143 | 0.6812 | 0.8000 |

In [ ]:
evaluation_report_path = EVALUATION_OUTPUT_DIR / "evaluation_report.csv"

evaluation_df = pd.read_csv(evaluation_report_path)
evaluation_df

## Results Discussion

The backend successfully generated all required output types for the three lecture videos. The summaries are concise, academic, and specific to each lecture topic.

The WER values are `0.0` because pseudo-reference transcripts were derived from the generated transcripts. These values confirm that the evaluation code is wired correctly, but they do not represent official human-reference transcription accuracy.

ROUGE values are populated using manually written reference summaries. Lecture 3 has the strongest ROUGE match, while Lecture 2 has lower ROUGE overlap because the generated summary uses different wording while still covering the relevant derivative and rate-of-change concepts.

## Video Duration Limitation

Based on SRT end timestamps, selected videos appear longer than the requested 10?15 minute target. This is documented as a limitation. The same pipeline can process shorter 10?15 minute videos.

## Limitations

- Official human transcripts were not available, so WER uses pseudo-reference transcripts.
- ROUGE depends on word overlap and does not fully measure conceptual correctness.
- Selected videos appear longer than the requested 10?15 minute target.
- Whisper can misrecognize mathematical terminology or lecture-specific wording.
- Subtitle timestamps are generated from Whisper segments and should be manually spot-checked for production use.
- FLAN-T5-base can produce weak summaries, so validation and fallback logic are included.

## Future Improvements

- Use official human transcripts for true WER evaluation.
- Add multiple human-written reference summaries for more reliable ROUGE evaluation.
- Add manual timestamp spot-checking for SRT quality assurance.
- Test the pipeline on shorter 10?15 minute lecture videos.
- Add more explicit topic segmentation for long lectures.
- Compare FLAN-T5-base with larger FLAN-T5, BART, or newer summarization models.
- Add automated tests for SRT formatting and summary validation.

## Conclusion

This notebook documents the current final backend pipeline for the NewtonAI Subtitle Generator and Summarizer project. The system extracts audio, transcribes lectures with Whisper, generates SRT subtitles, summarizes lecture content with FLAN-T5, and reports WER and ROUGE metrics.

The project is technically functional and ready for final packaging once the ZIP is created. The evaluation notes and video duration limitation are documented honestly so reviewers can interpret the results correctly.